# Generate augmentations of Dutch SeeGULL

This notebook generates augmented Dutch examples for the SeeGULL dataset using an LLM.
It produces three variants per example: `stereotype_sentence`, `neutral_sentence`, and `unrelated_sentence`,
then saves the augmented rows to a CSV for later merging and training.

## Overview
- Input: filtered SeeGULL CSV with `identity` and `attribute` columns
- Output: CSV with generated `stereotype`, `neutral` and `unrelated` sentence variants
- Model: external LLM (OpenAI) — responses must be valid JSON for easy parsing

In [ ]:
# Install minimal dependencies (uncomment and run if needed)
# !pip install -q openai pandas tqdm

In [ ]:
# Standard imports used in the notebook
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import json

## Configuration (set these before running)
- `OPENAI_API_KEY` should be exported in your shell or set securely.
- `INPUT_FILE` should point to the SeeGULL CSV to augment.
- `OUTPUT_FILE` will be created in the current folder.

In [ ]:
# Configuration
import os

# Prefer reading the API key from an environment variable
API_KEY = os.environ.get('OPENAI_API_KEY', '')
if not API_KEY:
    print('Warning: OPENAI_API_KEY not set; please export it before running the notebook.')

INPUT_FILE = 'DutchSeaGULL.csv'
OUTPUT_FILE = 'dutch_seagull_augmented_gpt4.csv'
MODEL = 'gpt-4o'  # Change if you use a different model

# Initialise client (the OpenAI client uses the provided key)
client = OpenAI(api_key=API_KEY)

In [ ]:
# 1. LOAD AND FILTER DATA
# Load the SeeGULL CSV and apply the same filtering used in the paper:
print('Loading data...')
df = pd.read_csv(INPUT_FILE)
print(f'Original size: {len(df)}')

# Filter 1: Consensus (Majority of annotators agree it is a stereotype)
df_consensus = df[df['stereotype'] >= 2].copy()

# Filter 2: Offensiveness (HEARTS removes scores <= 0)
df_filtered = df_consensus[df_consensus['mean_offensive_score'] > 0].copy()

print(f'Size after filtering (Consensus & Offensive > 0): {len(df_filtered)}')

Loading data...
Original size: 1233
Size after filtering (Consensus & Offensive > 0): 186


In [ ]:
# 2. DEFINE GPT FUNCTION
# This function calls the LLM and expects a JSON object with three fields.
def process_seagull_row(row):
    identity = row['identity']
    attribute = row['attribute']
    
    #ask for JSON format for easy parsing
    prompt = f"""
    You are a data augmentation expert for a Dutch stereotype detection dataset.
    
    Input Terms:
    - Identity: "{identity}"
    - Attribute (Stereotype): "{attribute}"
    
    Tasks:
    1. Create a short, natural Dutch sentence (5-10 words) linking the Identity and Attribute. (Label: "stereotype_sentence")
    2. Create a 'Neutral' variation: Replace the attribute with a neutral or positive word. Keep the Identity. (Label: "neutral_sentence")
    3. Create an 'Unrelated' variation: Replace the attribute with a random, nonsensical noun (e.g. 'fiets', 'kaas', 'tafel') so the sentence makes no sense. (Label: "unrelated_sentence")
    
    Output must be valid JSON only.
    Example JSON structure:
    {{
        "stereotype_sentence": "De Afghanen werden gezien als terroristen.",
        "neutral_sentence": "De Afghanen werden gezien als harde werkers.",
        "unrelated_sentence": "De Afghanen werden gezien als fietsen."
    }}
    """

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that outputs JSON."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.7
        )
        
        content = response.choices[0].message.content
        return json.loads(content)
    except Exception as e:
        print(f"Error processing {identity}: {e}")
        return None

In [ ]:
# 3. GENERATE AUGMENTATIONS
print('Starting generation with the LLM...')
results = []

# Iterate through filtered rows and call the model
for index, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    generated_data = process_seagull_row(row)

    if generated_data:
        new_row = row.to_dict()
        # Add generated fields to the row
        new_row['stereotype_sentence'] = generated_data.get('stereotype_sentence')
        new_row['neutral_sentence'] = generated_data.get('neutral_sentence')
        new_row['unrelated_sentence'] = generated_data.get('unrelated_sentence')
        results.append(new_row)

# Collect into a DataFrame and select output columns
df_final = pd.DataFrame(results)
cols = ['identity', 'attribute', 'mean_offensive_score', 'stereotype_sentence', 'neutral_sentence', 'unrelated_sentence']
cols += [c for c in df_final.columns if c not in cols]
df_final = df_final[cols]
# Save the augmented CSV
df_final.to_csv(OUTPUT_FILE, index=False)

print(f'Saved {len(df_final)} rows to {OUTPUT_FILE})')
print(df_final[['stereotype_sentence', 'neutral_sentence', 'unrelated_sentence']].head())

Starting generation with GPT-4...


100%|██████████| 186/186 [04:17<00:00,  1.38s/it]

Done! Saved 186 rows to dutch_seagull_augmented_gpt4.csv
                                 stereotype_sentence  \
0      Sommigen beschouwen Afghanen als terroristen.   
1   Rwandezen worden vaak geassocieerd met genocide.   
2  De Palestijnen worden vaak als terroristen gez...   
3       De Libische leider werd als dictator gezien.   
4            Koreanen worden vaak dictators genoemd.   

                                    neutral_sentence  \
0  Sommigen beschouwen Afghanen als vriendelijke ...   
1        Rwandezen staan bekend om hun gastvrijheid.   
2  De Palestijnen worden vaak als vredestichters ...   
3      De Libische leider werd als visionair gezien.   
4       Koreanen worden vaak als vriendelijk gezien.   

                              unrelated_sentence  
0      Sommigen beschouwen Afghanen als stoelen.  
1          Rwandezen staan bekend om hun tafels.  
2  De Palestijnen worden vaak als tafels gezien.  
3      De Libische leider werd als tafel gezien.  
4       Korean

### Problems encountered:
Too much repetition, the few shot prompt examples would be duplicated too closely.
Too much bias from the model, statements were intially generated with bias.